<a href="https://colab.research.google.com/github/lucas-estanislau/pe-de-meia/blob/Teste-de-otimiza%C3%A7%C3%A3o-da-coleta/01_coleta_inep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import requests
import zipfile
from io import BytesIO
import os
import re
from google.colab import drive
import unicodedata

!pip install xlrd -q
drive.mount("/content/drive")

Mounted at /content/drive


localiza as planilhas do Excel na pasta do projeto, identifica o ano de cada arquivo, organiza as planilhas em ordem crescente de ano e exibe seus nomes na tela.

In [2]:
PASTA = "/content/drive/MyDrive/Projeto_dados"

arquivos = {}
for f in os.listdir(PASTA):
    if f.lower().endswith((".xls", ".xlsx")):
        ano = int(re.search(r"(20\d{2})", f).group(1))
        arquivos[ano] = os.path.join(PASTA, f)

arquivos = dict(sorted(arquivos.items()))
for ano, caminho in arquivos.items():
    print(ano, os.path.basename(caminho))

2010 TAXAS RENDIMENTO UF 2010.xls
2011 tx_rendimento_uf_2011.xls
2012 tx_rendimento_UFs_2012.xlsx
2013 TAXAS RENDIMENTOS UF 2013.xlsx
2014 TAXAS RENDIMENTOS UF 2014.xlsx
2015 TX_REND_UFS_2015.xlsx
2016 TX_REND_UFS_2016.xlsx
2017 TX_REND_BRASIL_REGIOES_UFS_2017.xlsx
2018 TX_REND_BRASIL_REGIOES_UFS_2018.xlsx
2019 tx_rend_brasil_regioes_ufs_2019.xlsx
2020 tx_rend_brasil_regioes_ufs_2020.xlsx
2021 tx_rend_brasil_regioes_ufs_2021.xlsx
2022 tx_rend_brasil_regioes_ufs_2022.xlsx
2023 tx_rend_brasil_regioes_ufs_2023.xlsx
2024 tx_rend_brasil_regioes_ufs_2024.xlsx
2025 tx_rend_brasil_regioes_ufs_2025.xlsx


Dicionário com a sigla e o nome de cada uma das 27 UFs. É o que permite descartar Brasil e regiões, e padronizar tudo para sigla.

In [3]:
UFS = {"AC":"Acre","AL":"Alagoas","AP":"Amapá","AM":"Amazonas","BA":"Bahia","CE":"Ceará",
       "DF":"Distrito Federal","ES":"Espírito Santo","GO":"Goiás","MA":"Maranhão",
       "MT":"Mato Grosso","MS":"Mato Grosso do Sul","MG":"Minas Gerais","PA":"Pará",
       "PB":"Paraíba","PR":"Paraná","PE":"Pernambuco","PI":"Piauí","RJ":"Rio de Janeiro",
       "RN":"Rio Grande do Norte","RS":"Rio Grande do Sul","RO":"Rondônia","RR":"Roraima",
       "SC":"Santa Catarina","SP":"São Paulo","SE":"Sergipe","TO":"Tocantins"}

deixa o texto em minúsculas e sem acento, para comparar sem se preocupar com grafia.

In [10]:
def norm(x):
    """minúsculas, sem acento e sem espaços nas pontas"""
    x = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode()
    return x.strip().lower()

converte “Rondônia” ou “RO” em RO, por exemplo. Para Brasil, regiões ou qualquer outra coisa, devolve None, e é assim que essas linhas são descartadas.

In [11]:
SIGLAS = {norm(s): s for s in UFS}
NOMES = {norm(n): s for s, n in UFS.items()}

def para_sigla(v):
    n = norm(v)
    return SIGLAS.get(n) or NOMES.get(n)

converte a coluna em número. Valores como -- viram vazio (NaN).

In [12]:
def num(s):
    return pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")

Liga o nome da coluna final a uma palavra-chave que aparece no cabeçalho da planilha. A função principal usa essas palavras para achar a coluna certa em cada ano.

In [13]:
TAXAS = {"taxa_aprovacao": "aprova", "taxa_reprovacao": "reprova", "taxa_abandono": "abandon"}

 lê a planilha de cada ano, identifica as colunas necessárias, filtra os dados válidos por ano, estado, localização e dependência administrativa, seleciona as taxas desejadas, converte os valores para números e retorna uma tabela organizada para a análise de dados.

In [14]:
def ler_ano(caminho):
    bruto = pd.read_excel(caminho, sheet_name=0, header=None)


    ident = bruto.iloc[5:7, 1:6].apply(lambda c: norm(" ".join(c.dropna().astype(str))))

    def col_id(teste, nome):
        achadas = [c for c, r in ident.items() if teste(r)]
        if len(achadas) != 1:
            raise ValueError(f"coluna '{nome}': achei {len(achadas)} -> {ident.tolist()}")
        return achadas[0]

    c_uf = col_id(lambda r: r == "uf" or r.startswith("unidade"), "uf")
    c_loc = col_id(lambda r: "localiza" in r, "localizacao")
    c_dep = col_id(lambda r: "depend" in r or r == "rede", "dependencia")


    cab = bruto.iloc[5:8, 4:].copy()
    cab.iloc[0:2] = cab.iloc[0:2].ffill(axis=1)
    if cab.iloc[1].astype(str).str.contains("taxa", case=False).any():
        cab = cab.iloc[1:]
    rotulos = cab.apply(lambda c: norm(" ".join(c.dropna().astype(str))))


    d = bruto[pd.to_numeric(bruto[0], errors="coerce").between(2000, 2030)].copy()
    d["uf"] = d[c_uf].map(para_sigla)
    d = d[d["uf"].notna() & (d[c_loc].map(norm) == "total") & (d[c_dep].map(norm) == "total")]
    if d.empty:
        raise ValueError("nenhuma linha passou nos filtros de UF / localização / dependência")


    df = pd.DataFrame({"ano": pd.to_numeric(d[0]).astype(int), "uf": d["uf"]})
    colunas = {}
    for nome, chave in TAXAS.items():
        cand = [c for c, r in rotulos.items()
                if chave in r and "medio" in r and "total" in r and "nao" not in r]
        if len(cand) != 1:
            raise ValueError(f"'{chave}': esperava 1 coluna, achei {len(cand)} -> "
                             f"{[rotulos[c] for c in cand]}")
        colunas[nome] = cand[0]
        df[nome] = num(d[cand[0]])

    return df.reset_index(drop=True), colunas

Executando a função ler_ano em cada planilha e juntando os resultados em uma tabela só

In [15]:
dfs = []
for ano, caminho in arquivos.items():
    try:
        df, cols = ler_ano(caminho)
        dfs.append(df)
        print(f"{ano}: {len(df)} linhas | colunas usadas: {cols}")
    except Exception as e:
        print(f"{ano}: falhou -> {e}")

df_final = pd.concat(dfs, ignore_index=True)
df_final.head()

2010: 27 linhas | colunas usadas: {'taxa_aprovacao': 21, 'taxa_reprovacao': 39, 'taxa_abandono': 57}
2011: 27 linhas | colunas usadas: {'taxa_aprovacao': 16, 'taxa_reprovacao': 34, 'taxa_abandono': 52}
2012: 27 linhas | colunas usadas: {'taxa_aprovacao': 16, 'taxa_reprovacao': 34, 'taxa_abandono': 52}
2013: 27 linhas | colunas usadas: {'taxa_aprovacao': 16, 'taxa_reprovacao': 34, 'taxa_abandono': 52}
2014: 27 linhas | colunas usadas: {'taxa_aprovacao': 16, 'taxa_reprovacao': 34, 'taxa_abandono': 52}
2015: 27 linhas | colunas usadas: {'taxa_aprovacao': 16, 'taxa_reprovacao': 34, 'taxa_abandono': 52}
2016: 27 linhas | colunas usadas: {'taxa_aprovacao': 17, 'taxa_reprovacao': 35, 'taxa_abandono': 53}
2017: 27 linhas | colunas usadas: {'taxa_aprovacao': 16, 'taxa_reprovacao': 34, 'taxa_abandono': 52}
2018: 27 linhas | colunas usadas: {'taxa_aprovacao': 16, 'taxa_reprovacao': 34, 'taxa_abandono': 52}
2019: 27 linhas | colunas usadas: {'taxa_aprovacao': 16, 'taxa_reprovacao': 34, 'taxa_aband

,ano,uf,taxa_aprovacao,taxa_reprovacao,taxa_abandono
0,2010,RO,76.7,11.8,11.5
1,2010,AC,78.9,8.6,12.5
2,2010,AM,81.7,8.1,10.2
3,2010,RR,82.2,10.9,6.9
4,2010,PA,68.9,11.9,19.2


padronizar o texto em maiúsculo

In [19]:
df_final.columns = df_final.columns.str.upper()
df_final.head()

,ANO,UF,TAXA_APROVACAO,TAXA_REPROVACAO,TAXA_ABANDONO
0,2010,RO,76.7,11.8,11.5
1,2010,AC,78.9,8.6,12.5
2,2010,AM,81.7,8.1,10.2
3,2010,RR,82.2,10.9,6.9
4,2010,PA,68.9,11.9,19.2


Salvando no drive

In [20]:
PASTA_SAIDA = "/content/drive/MyDrive/inep_resultados"
os.makedirs(PASTA_SAIDA, exist_ok=True)

df_final.to_excel(os.path.join(PASTA_SAIDA, "taxas_em_uf_2010_2025.xlsx"), index=False)

print(os.listdir(PASTA_SAIDA))

['taxas_em_uf_2010_2025.xlsx']
